# Denoising Autoencoders With Keras

## Checking GPU Usage

Go to the `Runtime > Change runtime type` menu and check that we're on Python 3 and that the hardware accelerator is set to "GPU".

In [ ]:
!nvidia-smi

## Importing TensorFlow and the Other Required Libraries

In [ ]:
import matplotlib.pyplot as plt
import numpy
import tensorflow as tf
import keras
from keras import layers

## Loading MNIST

We'll use slightly different preprocessing than usual: since we want to be able to predict the input values at the output (the principle of autoencoding), we'll simply project these values into $[0, 1]$ instead of $[0, 255]$. Note that we usually don't do that: we normalize by centering on zero and dividing by the standard deviation.

In [ ]:
(X_train, _), (X_test, y_test) = keras.datasets.mnist.load_data()
nb_classes = 10
input_dim = 28 * 28
X_train = X_train.reshape(-1, input_dim) / 255.0
X_test = X_test.reshape(-1, input_dim) / 255.0

## Applying Gaussian Noise

In [ ]:
noise_factor = 0.5
X_train_noisy = X_train + numpy.random.normal(0, noise_factor, X_train.shape)
X_test_noisy = X_test + numpy.random.normal(0, noise_factor, X_test.shape)

# Clip the values to avoid pixels whiter than white (or blacker
# than black)
numpy.clip(X_train_noisy, 0, 1, out=X_train_noisy)
numpy.clip(X_test_noisy, 0, 1, out=X_test_noisy)

In [ ]:
n = 10
f, ax = plt.subplots(1, n, figsize=(n * 1.4, 2))
for i in range(n):
    ax[i].imshow(X_test_noisy[i].reshape(28, 28), cmap="gray_r")
    ax[i].set_title(y_test[i])
    ax[i].axis("off")
plt.show()

## Creating the Denoising Autoencoder

In [ ]:
# Your code here
encoding_dim = 12

autoencoder = keras.Sequential()

### Solution

In [ ]:
encoding_dim = 12

autoencoder = keras.Sequential(name="autoencoder")
autoencoder.add(layers.Input(shape=(input_dim,)))
autoencoder.add(layers.Dense(encoding_dim, activation="relu"))
autoencoder.add(layers.Dense(input_dim, activation="relu"))

autoencoder.summary()

autoencoder.compile(optimizer="adam", loss="mean_squared_error")

## Training

*Write the line that trains your autoencoder:*

- *50 epochs should be enough*
- *Use a batch size of 256*

In [ ]:
# Your code here

##Solution

In [ ]:
autoencoder.fit(X_train_noisy, X_train,
                epochs=50,
                batch_size=256,
                validation_split=0.2)

## Test Set

Autoencode the test images and store the resulting images in the variable `X_test_noisy_pred`

In [ ]:
# Your code here
X_test_noisy_pred = X_test_noisy

### Solution

In [ ]:
X_test_noisy_pred = autoencoder.predict(X_test_noisy)

## Visual Display of the Performance

In [ ]:
n = 10
_, ax = plt.subplots(2, n, figsize=(n * 1.4, 4))
for i, (ax_top, ax_bottom) in enumerate(ax.T):
    # The original on top
    ax_top.imshow(X_test_noisy[i].reshape(28, 28), cmap="gray_r")
    ax_top.set_title(str(y_test[i]))
    ax_top.axis("off")

    # The reconstruction at the bottom
    ax_bottom.imshow(X_test_noisy_pred[i].reshape(28, 28), cmap="gray_r")
    ax_bottom.axis("off")
plt.show()

## Try With More Neurons

What happens?

## Using Convolutional Networks

To do so, each image must be put back into a 28x28x1 shape. CNNs need this 3rd tensor dimension (there could be more channels than the gray level: there are 3 for color images, and even more in the intermediate layers of a convolutional network, where the number of input channels of a layer is the number of kernels of the previous layer).

In [ ]:
X_train = X_train.reshape(-1, 28, 28, 1)
X_test = X_test.reshape(-1, 28, 28, 1)
X_train_noisy = X_train_noisy.reshape(-1, 28, 28, 1)
X_test_noisy = X_test_noisy.reshape(-1, 28, 28, 1)

## Creating the Model

We'll use sequences of [`Conv2D`](https://keras.io/api/layers/convolution_layers/convolution2d/) and [`MaxPooling2D`](https://keras.io/api/layers/pooling_layers/max_pooling2d/) with kernels of 3 × 3 and 2 × 2 respectively for the encoder part. In these two layers, you'll need the `padding="same"` option to avoid border effects (the image being small enough as it is).

For the decoder part, we'll use [`Conv2D`](https://keras.io/api/layers/convolution_layers/convolution2d/) layers of the same kind followed by [`UpSampling2D`](https://keras.io/api/layers/reshaping_layers/up_sampling2d/) layers (of size 2 × 2), which are the inverse operation of [`MaxPooling2D`](https://keras.io/api/layers/pooling_layers/max_pooling2d/).

Feel free to overuse `model.summary()` to find your way. The goal is to get back a 28 × 28 × 1 image at the output of the last [`Conv2D`](https://keras.io/api/layers/convolution_layers/convolution2d/). Indeed, ending with an [`UpSampling2D`](https://keras.io/api/layers/reshaping_layers/up_sampling2d/) would be a bad idea.

The network will be deep: we'll use ReLU activation functions, except for the last one, where we'll use a sigmoid.

In [ ]:
# Your code here
autoencoder = keras.Sequential()

### Solution

In [ ]:
encoding_dim = 20


def conv(n, activation="relu"):
  return layers.Conv2D(n, (3, 3), activation=activation, padding="same")


def max_pool():
  return layers.MaxPooling2D((2, 2), padding="same")


def up_sampling():
  return layers.UpSampling2D((2, 2))


autoencoder = keras.Sequential(name="autoencoder")

# Encoding
autoencoder.add(keras.layers.Input(shape=X_train.shape[1:]))
autoencoder.add(conv(32))
autoencoder.add(max_pool())
autoencoder.add(conv(32))
autoencoder.add(max_pool())
autoencoder.add(conv(1))
autoencoder.add(layers.Flatten())
autoencoder.add(layers.Dense(encoding_dim))

# Decoding
autoencoder.add(layers.Dense(49))
autoencoder.add(layers.Reshape((7, 7, 1)))
autoencoder.add(conv(32))
autoencoder.add(up_sampling())
autoencoder.add(conv(32))
autoencoder.add(up_sampling())
autoencoder.add(conv(1, "sigmoid"))

autoencoder.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-4),
                    loss="binary_crossentropy")

autoencoder.summary()

## Training

*Write the line that trains your autoencoder:*

- *100 epochs should be enough*
- *Use a batch size of 256*

In [ ]:
# Your code here

### Solution

In [ ]:
autoencoder.fit(X_train_noisy, X_train,
                epochs=100,
                batch_size=128,
                validation_split=0.2)

## Displaying the Performance

In [ ]:
X_test_noisy_pred = autoencoder.predict(X_test_noisy).reshape(-1, 28, 28)

n = 10

random_indexes = numpy.random.choice(X_test.shape[0],
                                     replace=False,
                                     size=n)

_, ax = plt.subplots(2, n, figsize=(n * 1.4, 4))
for (ax_top, ax_bottom), random_index in zip(ax.T, random_indexes):
    # The original image on top
    ax_top.set_title(str(y_test[random_index]))
    ax_top.imshow(X_test_noisy[random_index].reshape(28, 28), cmap="gray_r")
    ax_top.axis("off")

    # The reconstructed image at the bottom
    ax_bottom.imshow(X_test_noisy_pred[random_index], cmap="gray_r")
    ax_bottom.axis("off")
plt.show()

## On Non-Noisy Images

In [ ]:
X_test_pred = autoencoder.predict(X_test).reshape(-1, 28, 28)

n = 10

random_indexes = numpy.random.choice(X_test.shape[0],
                                     replace=False,
                                     size=n)

_, ax = plt.subplots(2, n, figsize=(n * 1.4, 4))
for (ax_top, ax_bottom), random_index in zip(ax.T, random_indexes):
    # The original image on top
    ax_top.set_title(str(y_test[random_index]))
    ax_top.imshow(X_test[random_index].reshape(28, 28), cmap="gray_r")
    ax_top.axis("off")

    # The reconstructed image at the bottom
    ax_bottom.imshow(X_test_pred[random_index], cmap="gray_r")
    ax_bottom.axis("off")
plt.show()